# 04 — Structured Concurrency (TaskGroup)

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- utiliser `asyncio.TaskGroup` (Python 3.11+) pour la concurrence structurée ;
- comprendre les avantages par rapport à `gather()` ;
- gérer les `ExceptionGroup` avec `except*` ;
- utiliser `asyncio.timeout()` et `asyncio.timeout_at()` ;
- appliquer les patterns de concurrence structurée.

## Prérequis — ce que vous connaissez déjà

Vous maîtrisez déjà :

- `async def`, `await`, `create_task()`, `gather()` ;
- `wait()`, `wait_for()`, annulation de tasks ;
- les exceptions et `try/except`.

Notions introduites ici :

- `asyncio.TaskGroup` (PEP 654 / Python 3.11) ;
- `ExceptionGroup` et `except*` ;
- `asyncio.timeout()` (Python 3.11).

## Plan

1. Le problème avec `gather()`
2. `TaskGroup` — concurrence structurée
3. `ExceptionGroup` et `except*`
4. `asyncio.timeout()` — timeout structuré
5. Patterns avancés
6. Migration de `gather()` vers `TaskGroup`
7. Synthèse
8. Exercices

---

## 1. Le problème avec `gather()`

`gather()` a plusieurs problèmes :

1. **Fuite de tasks** : si une exception survient, les tasks restantes continuent en arrière-plan.
2. **Exceptions avalées** : avec `return_exceptions=True`, les exceptions sont silencieusement mises dans la liste.
3. **Pas de scope clair** : les tasks peuvent survivre à la fonction qui les a créées.

In [ ]:
import asyncio

async def ok() -> str:
    await asyncio.sleep(0.5)
    print("ok terminé (mais personne ne l'attend !)")
    return "ok"

async def echec() -> str:
    await asyncio.sleep(0.1)
    raise ValueError("boom")

# Avec gather : l'exception est levée, mais ok() continue en arrière-plan
try:
    await asyncio.gather(ok(), echec())
except ValueError as e:
    print(f"Exception : {e}")

await asyncio.sleep(1)  # ok() finit quand même en arrière-plan

---

## 2. `TaskGroup` — concurrence structurée

`asyncio.TaskGroup` (Python 3.11+, PEP 654) résout ces problèmes :

- Les tasks sont **scope-bound** : elles ne survivent pas au `async with`.
- En cas d'exception, toutes les tasks restantes sont **annulées**.
- Les exceptions sont collectées dans un `ExceptionGroup`.

In [ ]:
import asyncio

async def tache(nom: str, duree: float) -> str:
    await asyncio.sleep(duree)
    return f"{nom} OK"

async with asyncio.TaskGroup() as tg:
    t1 = tg.create_task(tache("A", 0.3))
    t2 = tg.create_task(tache("B", 0.5))
    t3 = tg.create_task(tache("C", 0.1))

# Ici, toutes les tasks sont terminées
print(f"A : {t1.result()}")
print(f"B : {t2.result()}")
print(f"C : {t3.result()}")

### Annulation automatique en cas d'erreur

In [ ]:
import asyncio

async def longue() -> str:
    try:
        await asyncio.sleep(10)
        return "longue terminée"
    except asyncio.CancelledError:
        print("longue() annulée par le TaskGroup")
        raise

async def echoue() -> str:
    await asyncio.sleep(0.1)
    raise RuntimeError("erreur !")

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(longue())
        tg.create_task(echoue())
except* RuntimeError as eg:
    for e in eg.exceptions:
        print(f"Exception capturée : {e}")

---

## 3. `ExceptionGroup` et `except*`

Python 3.11 introduit `ExceptionGroup` (PEP 654) pour représenter **plusieurs exceptions** levées simultanément. `except*` permet de les filtrer par type.

In [ ]:
# ExceptionGroup direct
eg = ExceptionGroup("erreurs multiples", [
    ValueError("mauvaise valeur"),
    TypeError("mauvais type"),
    ValueError("autre mauvaise valeur"),
])

try:
    raise eg
except* ValueError as matched:
    print(f"ValueError(s) : {matched.exceptions}")
except* TypeError as matched:
    print(f"TypeError(s) : {matched.exceptions}")

In [ ]:
import asyncio

async def fail_value() -> None:
    raise ValueError("val error")

async def fail_type() -> None:
    raise TypeError("type error")

async def fail_runtime() -> None:
    raise RuntimeError("runtime error")

try:
    async with asyncio.TaskGroup() as tg:
        tg.create_task(fail_value())
        tg.create_task(fail_type())
        tg.create_task(fail_runtime())
except* ValueError as eg:
    print(f"ValueErrors : {[str(e) for e in eg.exceptions]}")
except* (TypeError, RuntimeError) as eg:
    print(f"Type/Runtime : {[str(e) for e in eg.exceptions]}")

---

## 4. `asyncio.timeout()` — timeout structuré

Python 3.11 ajoute `asyncio.timeout()` comme context manager, plus propre que `wait_for()`.

In [ ]:
import asyncio

async def lente() -> str:
    await asyncio.sleep(5)
    return "fini"

try:
    async with asyncio.timeout(1.0):
        resultat = await lente()
except TimeoutError:
    print("Timeout après 1s !")

In [ ]:
import asyncio

# Combiner TaskGroup et timeout
try:
    async with asyncio.timeout(1.0):
        async with asyncio.TaskGroup() as tg:
            tg.create_task(asyncio.sleep(0.5))
            tg.create_task(asyncio.sleep(5))  # trop long
except TimeoutError:
    print("TaskGroup annulé par timeout")

### `timeout_at()` — deadline absolue

In [ ]:
import asyncio

loop = asyncio.get_running_loop()
deadline = loop.time() + 1.0  # dans 1 seconde

try:
    async with asyncio.timeout_at(deadline):
        await asyncio.sleep(5)
except TimeoutError:
    print("Timeout par deadline absolue")

---

## 5. Patterns avancés

### 5.1. Fan-out / Fan-in avec TaskGroup

In [ ]:
import asyncio

async def traiter(item: int) -> int:
    await asyncio.sleep(0.1)
    return item * item

resultats: list[int] = []

async def collecter(tg: asyncio.TaskGroup, item: int) -> None:
    r = await traiter(item)
    resultats.append(r)

async with asyncio.TaskGroup() as tg:
    for i in range(10):
        tg.create_task(collecter(tg, i))

print(f"Résultats : {sorted(resultats)}")

### 5.2. TaskGroup imbriqués

In [ ]:
import asyncio

async def sous_tache(nom: str) -> str:
    await asyncio.sleep(0.1)
    return f"{nom} OK"

async def groupe(nom_groupe: str) -> list[str]:
    resultats = []
    async with asyncio.TaskGroup() as tg:
        tasks = [
            tg.create_task(sous_tache(f"{nom_groupe}-{i}"))
            for i in range(3)
        ]
    return [t.result() for t in tasks]

async with asyncio.TaskGroup() as tg:
    t_a = tg.create_task(groupe("GroupeA"))
    t_b = tg.create_task(groupe("GroupeB"))

print(f"A : {t_a.result()}")
print(f"B : {t_b.result()}")

---

## 6. Migration de `gather()` vers `TaskGroup`

| `gather()` | `TaskGroup` |
|---|---|
| `results = await gather(c1, c2)` | `async with TaskGroup() as tg:` |
| `return_exceptions=True` | `except* ExType` |
| Tasks fuient en cas d'erreur | Annulation automatique |
| Résultats dans l'ordre | Via `task.result()` |

**Recommandation :** préférez `TaskGroup` pour tout nouveau code (Python 3.11+). `gather()` reste utile pour les one-liners simples sans gestion d'erreur complexe.

In [ ]:
import asyncio

# Avant (gather)
async def avant() -> list[int]:
    return await asyncio.gather(
        asyncio.sleep(0.1, result=1),
        asyncio.sleep(0.1, result=2),
        asyncio.sleep(0.1, result=3),
    )

# Après (TaskGroup)
async def apres() -> list[int]:
    async with asyncio.TaskGroup() as tg:
        tasks = [
            tg.create_task(asyncio.sleep(0.1, result=i))
            for i in [1, 2, 3]
        ]
    return [t.result() for t in tasks]

print(f"gather   : {await avant()}")
print(f"TaskGroup: {await apres()}")

---

## 7. Synthèse

| Outil | Usage |
|---|---|
| `TaskGroup` | Concurrence structurée : scope clair, annulation auto |
| `tg.create_task()` | Ajouter une task au groupe |
| `ExceptionGroup` | Plusieurs exceptions simultanées |
| `except*` | Filtrer par type dans un ExceptionGroup |
| `asyncio.timeout(s)` | Timeout structuré |
| `asyncio.timeout_at(t)` | Timeout par deadline absolue |

**Concurrence structurée = 3 garanties :**

1. Toutes les tasks sont terminées quand on sort du `async with`.
2. En cas d'erreur, les tasks restantes sont annulées.
3. Toutes les exceptions sont collectées et exposées.

---

## 8. Exercices

### Exercice 1 — TaskGroup basique *(facile)*

Réécrire l'exemple classique de 5 "téléchargements" simulés en utilisant `TaskGroup` au lieu de `gather()`. Afficher les résultats.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Structured_concurrency", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import time

async def telecharger(nom: str) -> str:
    await asyncio.sleep(0.3)
    return f"{nom}: OK"

start = time.perf_counter()
async with asyncio.TaskGroup() as tg:
    tasks = [tg.create_task(telecharger(f"file_{i}")) for i in range(5)]

for t in tasks:
    print(f"  {t.result()}")
print(f"Temps : {time.perf_counter() - start:.2f}s")
```

</details>

### Exercice 2 — Gestion d'erreurs avec except* *(moyen)*

Créer un TaskGroup avec 6 tasks : les tasks paires lèvent `ValueError`, les impaires retournent normalement. Utiliser `except*` pour capturer les `ValueError` et afficher le nombre d'erreurs.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Structured_concurrency", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio

async def tache(i: int) -> int:
    await asyncio.sleep(0.05)
    if i % 2 == 0:
        raise ValueError(f"erreur-{i}")
    return i

try:
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(tache(i)) for i in range(6)]
except* ValueError as eg:
    print(f"Nombre de ValueError : {len(eg.exceptions)}")
    for e in eg.exceptions:
        print(f"  {e}")
```

</details>

### Exercice 3 — Scraper structuré avec timeout *(difficile)*

Écrire un scraper qui :

1. Utilise un `TaskGroup` pour lancer 10 requêtes simulées.
2. Le tout dans un `asyncio.timeout(2.0)`.
3. 3 des requêtes prennent 5s (seront annulées par le timeout).
4. Capturer le `TimeoutError` et afficher combien de tasks ont réussi.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Structured_concurrency", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import random

async def requete(i: int) -> str:
    duree = 5.0 if i < 3 else random.uniform(0.1, 0.5)
    await asyncio.sleep(duree)
    return f"req-{i}: OK"

tasks = []
try:
    async with asyncio.timeout(2.0):
        async with asyncio.TaskGroup() as tg:
            tasks = [tg.create_task(requete(i)) for i in range(10)]
except TimeoutError:
    reussies = [t for t in tasks if t.done() and not t.cancelled() and t.exception() is None]
    print(f"Timeout ! {len(reussies)}/{len(tasks)} réussies")
    for t in reussies:
        print(f"  {t.result()}")
```

</details>

---

## Ressources

- [docs Python — asyncio.TaskGroup](https://docs.python.org/3/library/asyncio-task.html#asyncio.TaskGroup)
- [PEP 654 — Exception Groups and except*](https://peps.python.org/pep-0654/)
- [Notes on Structured Concurrency](https://vorpus.org/blog/notes-on-structured-concurrency-or-go-statement-considered-harmful/) — Nathaniel Smith
- [Trio](https://trio.readthedocs.io/) — bibliothèque qui a inspiré TaskGroup